In [1]:
import socket
import threading
import time

clients = []

def handle_server_client(conn):
    while True:
        try:
            data = conn.recv(1024)
            if not data:
                break
            for c in clients:
                if c != conn:
                    c.send(data)
        except Exception:
            break
    if conn in clients:
        clients.remove(conn)
    conn.close()

def run_server():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(('127.0.0.1', 5000))
    s.listen()
    print("[Server] Started and listening on port 5000...")
    while True:
        conn, _ = s.accept()
        clients.append(conn)
        threading.Thread(target=handle_server_client, args=(conn,), daemon=True).start()

threading.Thread(target=run_server, daemon=True).start()
time.sleep(1)

def listen_to_server(sock, client_name):
    while True:
        try:
            msg = sock.recv(1024)
            if not msg:
                break
            print(f"\n[{client_name} Received]: {msg.decode()}")
        except Exception:
            break

client_a = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_a.connect(('127.0.0.1', 5000))
threading.Thread(target=listen_to_server, args=(client_a, "Client A"), daemon=True).start()

client_b = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_b.connect(('127.0.0.1', 5000))
threading.Thread(target=listen_to_server, args=(client_b, "Client B"), daemon=True).start()

time.sleep(0.5)

print("\n--- Sending test messages across sockets ---")
client_a.send("Hello from Client A!".encode())
time.sleep(1)

client_b.send("Hey Client A, got your message!".encode())
time.sleep(1)

msg = input("Type a message to send from Client A (or press Enter to skip): ")
if msg.strip():
    client_a.send(msg.encode())
    time.sleep(1)

[Server] Started and listening on port 5000...

--- Sending test messages across sockets ---

[Client B Received]: Hello from Client A!

[Client A Received]: Hey Client A, got your message!
Type a message to send from Client A (or press Enter to skip): hi

[Client B Received]: hi
